# Portfolio Risk Decomposition — Construction Spec (USE4-faithful)

**Purpose.** This notebook is the **source-of-truth spec** for building
`risk_decomp_build.ipynb` — the risk model's **payoff and first end-to-end test**: it
composes the exposures `X` (68 factors), the factor covariance `F` (built in `06_fcov`),
and the specific diagonal `Δ` (built in `07_specific_risk`) into portfolio risk and the
standard Barra/USE4 attribution.

```
σ²_p = wᵀ X F Xᵀ w  +  wᵀ Δ w        (factor + specific)
```

It is a *consumer* of the model (it takes a portfolio `w`), built in the same stage
pattern as every other section. Implement the four decomposition kernels — the
stacked→dense `F` rebuild (`dense_F`), the decomposition itself (`decompose`), the
group aggregation (`group_contrib`), and the bias statistic (`bias_stat`) — as pure
numpy functions in a module of your own, and validate them with a known-answer test
script (synthetic data with known truth) **before** wiring them into the build
notebook (§4 enumerates the tests). The build notebook then applies the kernels to
ship two artifacts and validate end-to-end.

**Audience.** You — plus whatever AI assistant you are driving. Treat its output as
deeply untrustworthy until audited. Follow the stages linearly.

**Companion docs.** The risk-decomposition chapter in `textbook/` carries the reference numbers; upstream specs
`06_fcov/fcov_spec.ipynb` and `07_specific_risk/specific_risk_spec.ipynb`.

> **Your task.** The spec is provided (the risk-decomposition chapter in `textbook/` carries the reference numbers);
> `risk_decomp_build.ipynb` is yours to write. **This stage runs LAST** — it consumes
> deliverables from every one of sections 04–07, so finish those builds first.

## §1. Deliverables and output schema

Both artifacts are written with `compression="zstd"`, `statistics=True`.

**Asset total risk** — `data/out/asset_total_risk.parquet` (per coverage stock-date;
1,044,527 rows over 288 common month-ends on the reference run):

| Column | Type | Description |
|---|---|---|
| `permaticker` | Int64 | stock ID |
| `signal_date` | Date | forecast (month-end) date t |
| `in_estu` | Bool | ESTU vs non-ESTU coverage |
| `factor_risk` | Float64 | √(xᵢᵀ F xᵢ) — systematic vol of the stock |
| `specific_risk` | Float64 | σ_spec,i |
| `total_risk` | Float64 | √(factor² + specific²) — predicted total vol (monthly) |

This is the diagonal of `X F Xᵀ + Δ` — every coverage stock's predicted volatility,
with no portfolio required. Non-ESTU stocks appear here (`in_estu = False`) but never
enter a test portfolio.

**Portfolio decomposition** — `data/out/portfolio_risk_decomp.parquet`, one row per
(`signal_date`, `portfolio`) — 5 portfolios × 288 dates = 1,440 rows on the reference
run:

| Column | Type | Description |
|---|---|---|
| `signal_date` | Date | forecast (month-end) date t |
| `portfolio` | String | `market` / `equal_weight` / `mom_tilt` / `value_tilt` / `size_tilt` |
| `total_risk`, `factor_risk`, `specific_risk` | Float64 | σ_p and its factor/specific split (monthly vols) |
| `active_te` | Float64 | tracking error vs the market (0.0 for the market itself, by construction) |
| `ctr_country`, `ctr_industry`, `ctr_style`, `ctr_specific` | Float64 | factor-group risk contributions — the four **sum to σ_p** |
| `realized_ret` | Float64 | realized portfolio return over (t, t+1], for the bias statistic |

## §2. Pipeline

```
STAGE 0  Setup, parameters (test-portfolio definitions, winsor cap)
STAGE 1  Inputs: canonical 68-factor order (from data/out/csr_factor_returns.parquet);
         F (data/out/fcov_factor_cov.parquet — stacked upper triangle, reconstruct the
         68×68 per est_date); Δ (data/out/specific_risk.parquet, sigma_spec); the full
         exposure matrix X (data/out/industries_use4.parquet one-hot + the 12 style
         files); realized returns (data/out/csr_specific_returns.parquet, column y)
STAGE 2  Per common date: assemble X (n×68) + F (68×68) + dvar; compute asset total
         risk diag(XFXᵀ)+Δ; build the canonical portfolios; decompose each
         (total/factor/specific, factor-group CTR, active TE vs market) + realized return
STAGE 3  Ship asset_total_risk.parquet + portfolio_risk_decomp.parquet
STAGE 4  Validate (identity, Euler additivity, market≈country, bias stats, tilt attribution)
```

```python
# ───── Parameters ─────────────────────────────────────────────────────────────
ANN   = sqrt(12)   # monthly vol -> annualized (reporting only)
Z_CAP = 10.0       # winsorize the portfolio bias statistic — same project-wide cap
                   # as the factor-covariance and specific-risk internal statistics

# pure style-tilt test portfolios (dollar-neutral, gross ~2): name -> score column
TILTS = {"mom_tilt": "mom_score", "value_tilt": "btop_score", "size_tilt": "size_score"}
UNASSIGNED = "UNASSIGNED (fallback ladder)"   # industry label dropped from the universe
```

**Position.** Runs **last** — the full order is `04 country → 05 csr_build →
05 daily_csr_build → 06 fcov_build → 07 specific_risk_build → 08 risk_decomp_build`.
Common dates = `F ∩ Δ` — the 288 month-ends 2002-04-30 → 2026-03-31 on the reference
run (`F`'s warm-up binds; the window extends as upstream history grows). A date present
in `Δ` but not in `F` is simply excluded by the intersection. Reuse the canonical
factor order `['country'] + sorted(industries) + sorted(styles)` (1 + 55 + 12 = 68,
derived from `csr_factor_returns.parquet` and asserted == 68) so X's columns align
with F **by name**; assert the stacked F's factor names are a subset of the canonical 68.

**X is the full 68-column form** — country≡1, 55 industry one-hot dummies, 12 style
scores — NOT the CSR's constraint-reparametrized 56-column regression design. The
decomposition needs every factor's loading; identification (which mattered for
*estimating* the factor returns) is irrelevant for *evaluating* `xᵀFx`.

**Universe.** Per date: the specific-risk coverage panel (`permaticker, signal_date,
in_estu, sigma_spec, mcap`), left-joined to the industry label and the 12 style scores
(note `btop_score` lives in `bp_use4.parquet`, `mom_score` in `mom_use4.parquet`),
keeping stocks with a classified industry (drop `UNASSIGNED`/null — 28 stock-dates on
the reference run) and complete-case on all 12 style columns (if you later choose to
impute missing exposures, this join is where it would land).

## §3. The decomposition (standard Barra)

For a portfolio `w` with factor exposures `x = Xᵀw`:

- **Total** `σ_p = √(xᵀF x + Σ wᵢ²σ²_spec,i)`; report the factor / specific split.
- **Asset contributions** (Euler — `σ_p` is homogeneous degree 1 in `w`):
  `MCTRᵢ = (X F x + Δw)ᵢ / σ_p`, `CTRᵢ = wᵢ·MCTRᵢ`, and **`Σ CTRᵢ = σ_p`** exactly.
- **Factor contributions** `CTRₖ = xₖ (F x)ₖ / σ_p` — the Barra "exposure × vol ×
  correlation" form `xₖ·σ_fₖ·ρ(fₖ,R_p)`; `Σₖ CTRₖ + V_specific/σ_p = σ_p`. Grouped into
  country / industry / style.
- **Active risk** `w_active = w_p − w_b`, `b` = cap-weighted ESTU (the model's market,
  corr(f_country, market) ≈ 0.998); identical formulas → tracking error + attribution.

Never form the dense N×N asset covariance — keep everything as `X F Xᵀ + Δ`. Asset
total risk uses `diag(XFXᵀ)ᵢ = einsum('ij,ij->i', X@F, X)` (clamp `max(·, 0)` before the
square root — non-PSD numerical noise gets floored, never propagated); the decompose
kernel takes `(w, X, F, dvar)` and evaluates `X F (Xᵀw) + Δw` without ever materializing
the N×N matrix.

Two silent-failure warnings, worth keeping in front of you the whole build:

1. **F arrives as a stacked upper triangle** (`est_date, factor_i, factor_j, cov`) —
   when rebuilding the dense 68×68 you must fill **both** `F[i,j]` **and** `F[j,i]`.
   Forgetting the mirror zeroes half of every off-diagonal term and nothing crashes.
   Guard the rebuild: assert the per-date entry count is exactly K(K+1)/2 (triangle)
   or K² (full) and that every entry is finite.
2. **The decompose kernel consumes specific *variances*** — `dvar = sigma_spec²`.
   Square `sigma_spec` before passing; feeding vols is silently wrong (no shape or
   sign error will save you).

Numerical guards in the kernel: `σ_p = √(max(var_f + var_s, 0))`, and when `σ_p = 0`
return zero-filled contributions instead of dividing by zero. The zero branch is
exercised **in production on every date** — the market's active TE against itself is
exactly 0 (`w − w_b = 0`).

**Test portfolios** (built per date on `in_estu & (mcap > 0)`): cap-weighted (the
market, `w = mcap/Σmcap`), equal-weight (`1/n_estu`), and dollar-neutral style
long-shorts (momentum, value/btop, size): `w ∝ score`, zeroed outside ESTU, centered
by the ESTU mean (dollar-neutral), scaled by `0.5·Σ|s|` → gross ≈ 2 (1 long / 1
short); an all-zero score date degenerates safely to all-zero weights. They double as
validation — the market should be ~all country, a style L/S ~all that style.

## §4. Validation contract

| check | gate |
|---|---|
| 1 | asset total risk positive, finite; `factor² + specific² = total²` per row to 1e-12 (observed max residual 4.4×10⁻¹⁶) |
| 2 | **Euler additivity** — re-decompose the market on the latest date: asset CTRs sum to σ_p to 1e-10; factor-group CTRs sum to `total_risk` to 1e-9 (observed Σactr−σ_p = 0.0) |
| 3 | the cap-weighted ESTU **market ≈ the country factor** — median annualized vol ∈ (10, 25)% and median factor variance share > 0.90 (observed 14.5%/yr, 99.3%) |
| 4 | country dominates market risk — median country CTR share of σ_p > 0.80 (observed 99%) |
| 5 | **portfolio bias statistics ~ 1** — predicted σ_p,t vs realized return over (t,t+1], winsorized ±`Z_CAP`: market ∈ (0.8, 1.2) and every portfolio ∈ (0.7, 1.3) (observed: market 0.908, equal_weight 0.935, mom_tilt 0.776, value_tilt 0.857, size_tilt 0.797) |
| 6 | attribution sanity — each style tilt's risk is dominated by the style group: median style-group CTR share of σ_p > 0.45 (observed mom 81%, value 55%, size 52%) |

Checks 2–5 are the load-bearing results: additivity certifies the math, the
market≈country result and the style-tilt attribution certify that the model attributes
risk to the right places, and the bias statistic is the **first test that X, F and Δ
compose** into a correct portfolio risk forecast. Round the battery out with a
factor-group attribution figure (stacked group-CTR shares of σ_p over time, per
portfolio) as an informational diagnostic.

Informational (print, don't gate): realized-return coverage — holdings with a null `y`
are excluded from `realized_ret`, and the minimum gross-weight coverage across
portfolio-dates is tracked so the gap is never silent (observed min 99.28%); read-back
row-count asserts on both written parquets; a latest-date annualized decomposition
snapshot (σ, factor/specific split, TE, group shares per portfolio).

**Kernel known-answer tests (write these first).** Before the build notebook touches
real data, your test script must pass on synthetic data with known truth (fixed seed):

1. **dense-F round-trip** — stack the upper triangle of a random PSD K=8 matrix, rebuild,
   and require `allclose` to the original and exact symmetry.
2. **Euler additivity** — random portfolio (n=300, K=12, unit gross): `Σ actr = σ_p` and
   `Σ fctr + sctr = σ_p` to 1e-10; `var_factor + var_specific = σ_p²` to 1e-12.
3. **Single-factor bet** — `X = I`, `dvar = 0`, `w = e_k`: `σ_p = √F_kk` to 1e-12, and the
   factor CTR concentrates entirely on factor k.
4. **All-specific case** — `F = 0`: `σ_p = √(Σ w²·dvar)` and `var_factor ≈ 0`.
5. **Group partition** — with `factor_type = [country, industry×3, style×2]`, the three
   group CTRs plus specific sum to σ_p to 1e-10.
6. **Bias statistic** — `bias_stat ≈ 1` (±0.02) for `realized = pred·N(0,1)` over 50,000
   draws; then plant a single 500σ blow-up and require the ±`Z_CAP` winsor to keep the
   statistic ≈ 1.

## §5. Master list of decisions

The decomposition math is standard Barra; the implementation calls:

| # | Decision | Default chosen | Where to revisit |
|---|---|---|---|
| 1 | Exposure form | Full 68-col X (not the reparametrized 56-col CSR design) | Never — every factor's loading is needed |
| 2 | Benchmark (active risk) | Cap-weighted ESTU (the model's market) | If a published index benchmark is required |
| 3 | Coverage policy | Drop UNASSIGNED-industry stocks (28 stock-dates) | If uncovered weight in a real book grows material |
| 4 | Common dates | F ∩ Δ = 2002-04-30 → 2026-03-31 (F warm-up binds) | Self-heals as F's history extends |
| 5 | Test portfolios | cap-wtd ESTU, equal-weight, mom/value/size long-shorts | Add a real holdings book when supplied |
| 6 | Bias-stat winsorization | Standardized return clipped at ±10 (same cap as the model's internal stats) | If a less/more robust metric is wanted |

The portfolio biases sit slightly below 1 (market 0.91, tilts 0.78–0.86) — a mild,
in-band over-prediction inherited from the factor model's ~0.95 calibration (the
dollar-neutral tilts are near-pure factor bets, so they carry that bias most). With
this stage the risk model is **usable end-to-end**.